# Variaveis que podem ser utilizadas

1. Adequação do nível (IAN)

2. Desempenho acadêmico (IDA)

3. Engajamento nas atividades (IEG)

4. Autoavaliação (IAA)

5. Aspectos psicossociais (IPS)

6. Aspectos psicopedagógicos (IPP)

7. Ponto de virada (IPV)

8. Indice nacional de desenvolvimento (INDE)

adicionar sexo, defasagem,

In [328]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import streamlit as st
import plotly.express as px

In [329]:
arquivo = r"C:\Users\gabri\Desktop\FIAP\Fase_5\Tech_Challenge\data\BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

In [330]:
def limpar_colunas(df):
    # remove espaços invisíveis
    df.columns = df.columns.str.strip()

    # remove duplicadas mantendo a primeira
    df = df.loc[:, ~df.columns.duplicated()]

    return df

In [331]:
def padronizar(df, ano):

    df = limpar_colunas(df)

    # mantém apenas o INDE do ano atual
    col_inde = [
    c for c in df.columns
    if "INDE" in c and (str(ano)[-2:] in c or str(ano) in c)
    ]

    if len(col_inde) > 0:
        df["INDE"] = df[col_inde[0]]
    else:
        df["INDE"] = None

    mapa = {
        "Nome": "NOME",
        "Nome Anonimizado": "NOME",
        "Gênero":"SEXO",
        "Idade 22": "IDADE",
        "Idade": "IDADE",
        "IAN": "IAN",
        "IDA": "IDA",
        "IEG": "IEG",
        "IAA": "IAA",
        "IPS": "IPS",
        "IPP": "IPP",
        "IPV": "IPV",
        "Defas": "DEFASAGEM",
        "Defasagem": "DEFASAGEM"
    }

    df = df.rename(columns=mapa)

    COLUNAS_PADRAO = [
        "RA", "NOME", "SEXO", "IDADE", "DEFASAGEM",
        "IAN", "IDA", "IEG", "IAA",
        "IPS", "IPP", "IPV", "INDE"
    ]

    for col in COLUNAS_PADRAO:
        if col not in df.columns:
            df[col] = None

    df = df[COLUNAS_PADRAO].copy()
    df["ANO"] = ano

    return df


In [332]:
df_2022 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2022"), 2022)
df_2023 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2023"), 2023)
df_2024 = padronizar(pd.read_excel(arquivo, sheet_name="PEDE2024"), 2024)

In [333]:
for nome, df_temp in {
    "2022": df_2022,
    "2023": df_2023,
    "2024": df_2024
}.items():

    duplicadas = df_temp.columns[df_temp.columns.duplicated()]
    print(nome, "-> duplicadas:", list(duplicadas))


2022 -> duplicadas: []
2023 -> duplicadas: []
2024 -> duplicadas: []


In [334]:
df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

In [335]:
df.head()

,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO
0,RA-1,Aluno-1,Menina,19,-1,5.0,4.0,4.1,8.3,5.6,None,7.278,5.783,2022
1,RA-2,Aluno-2,Menina,17,0,10.0,6.8,5.2,8.8,6.3,None,6.778,7.055,2022
2,RA-3,Aluno-3,Menina,17,0,10.0,5.6,7.9,0.0,5.6,None,7.556,6.591,2022
3,RA-4,Aluno-4,Menino,17,0,10.0,5.0,4.5,8.8,5.6,None,5.278,5.951,2022
4,RA-5,Aluno-5,Menina,17,0,10.0,5.2,8.6,7.9,5.6,None,7.389,7.427,2022


In [336]:
df["SEXO"].unique()

array(['Menina', 'Menino', 'Feminino', 'Masculino'], dtype=object)

In [337]:
#padroniza a coluna sexo
mapa_sexo = {
    "Menina": "F",
    "Feminino": "F",
    "Menino": "M",
    "Masculino": "M"
}

df["SEXO"] = df["SEXO"].map(mapa_sexo)


In [338]:
df["SEXO"].value_counts()

SEXO
F    1626
M    1404
Name: count, dtype: int64

In [339]:
#converte para números para usar a coluna sexo como feature no modelo
df["SEXO"] = df["SEXO"].map({"F": 0, "M": 1})

In [340]:
df.columns

Index(['RA', 'NOME', 'SEXO', 'IDADE', 'DEFASAGEM', 'IAN', 'IDA', 'IEG', 'IAA',
       'IPS', 'IPP', 'IPV', 'INDE', 'ANO'],
      dtype='object')

In [341]:
cols = ['IDADE','DEFASAGEM','IAN','IDA','IEG','IAA','IPS','IPP','IPV','INDE', 'ANO']

# verifica se as colunas possuem algum valor que não sejam números
for c in cols:
    erro = pd.to_numeric(df[c], errors='coerce').isna() & df[c].notna()
    print(c, "-> valores inválidos:", erro.sum(), "-> valor identificado: ", df.loc[erro, c].unique())

IDADE -> valores inválidos: 399 -> valor identificado:  [datetime.datetime(1900, 1, 8, 0, 0) datetime.datetime(1900, 1, 7, 0, 0)
 datetime.datetime(1900, 1, 11, 0, 0) datetime.datetime(1900, 1, 9, 0, 0)
 datetime.datetime(1900, 1, 10, 0, 0) datetime.datetime(1900, 1, 14, 0, 0)
 datetime.datetime(1900, 1, 13, 0, 0) datetime.datetime(1900, 1, 12, 0, 0)
 datetime.datetime(1900, 1, 15, 0, 0) datetime.datetime(1900, 1, 17, 0, 0)
 datetime.datetime(1900, 1, 16, 0, 0) datetime.datetime(1900, 1, 19, 0, 0)
 datetime.datetime(1900, 1, 18, 0, 0) datetime.datetime(1900, 1, 20, 0, 0)
 datetime.datetime(1900, 1, 21, 0, 0) datetime.datetime(1900, 1, 26, 0, 0)]
DEFASAGEM -> valores inválidos: 0 -> valor identificado:  []
IAN -> valores inválidos: 0 -> valor identificado:  []
IDA -> valores inválidos: 0 -> valor identificado:  []
IEG -> valores inválidos: 0 -> valor identificado:  []
IAA -> valores inválidos: 0 -> valor identificado:  []
IPS -> valores inválidos: 0 -> valor identificado:  []
IPP -> val

In [342]:
# Converte para NaN os valores incorretos da coluna INDE
df['INDE'] = pd.to_numeric(df['INDE'], errors='coerce')
# Converte para NaN os valores incorretos da coluna IDADE
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')

In [343]:
# Preenche com 0 os valores NaN
cols = ['IDADE','DEFASAGEM','IAN','IDA','IEG','IAA','IPS','IPP','IPV', 'INDE']

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols] = df[cols].fillna(df[cols].median())

In [344]:
# Converte para int a coluna ANO
df['ANO'] = pd.to_numeric(df['ANO'], errors='coerce').astype('Int64')
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce').astype('Int64')

In [345]:
df[cols].dtypes

IDADE          Int64
DEFASAGEM      int64
IAN          float64
IDA          float64
IEG          float64
IAA          float64
IPS          float64
IPP          float64
IPV          float64
INDE         float64
dtype: object

In [346]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   RA         3030 non-null   object 
 1   NOME       3030 non-null   object 
 2   SEXO       3030 non-null   int64  
 3   IDADE      3030 non-null   Int64  
 4   DEFASAGEM  3030 non-null   int64  
 5   IAN        3030 non-null   float64
 6   IDA        3030 non-null   float64
 7   IEG        3030 non-null   float64
 8   IAA        3030 non-null   float64
 9   IPS        3030 non-null   float64
 10  IPP        3030 non-null   float64
 11  IPV        3030 non-null   float64
 12  INDE       3030 non-null   float64
 13  ANO        3030 non-null   Int64  
dtypes: Int64(2), float64(8), int64(2), object(2)
memory usage: 337.4+ KB


In [347]:
df = df.sort_values(["RA", "ANO"])

df["delta_IDA"] = df.groupby("RA")["IDA"].diff()
df["delta_INDE"] = df.groupby("RA")["INDE"].diff()

In [348]:
df.head(10)

,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE
0,RA-1,Aluno-1,0,19,-1,5.0,4.000000,4.100000,8.300,5.60,7.500000,7.278,5.783000,2022,NaN,NaN
1855,RA-1,Aluno-1,0,12,0,10.0,6.666667,8.600000,8.751,7.50,7.500000,7.583,7.388267,2023,2.666667,1.605267
2958,RA-1,Aluno-1,0,21,0,10.0,6.666667,0.000000,8.751,7.50,7.500000,7.583,7.388267,2024,0.000000,0.000000
9,RA-10,Aluno-10,0,18,-1,5.0,4.100000,5.200000,8.300,5.00,7.500000,7.056,5.784000,2022,NaN,NaN
99,RA-100,Aluno-100,0,13,1,10.0,7.600000,7.800000,8.800,5.00,7.500000,7.250,7.618000,2022,NaN,NaN
1072,RA-1000,Aluno-1000,0,8,0,10.0,7.000000,9.400000,8.500,3.77,6.250000,8.920,7.916200,2023,NaN,NaN
2225,RA-1000,Aluno-1000,0,9,0,10.0,7.750000,9.545455,9.002,6.26,8.125000,7.835,8.364791,2024,0.750000,0.448591
1074,RA-1001,Aluno-1001,0,9,-1,5.0,7.800000,9.100000,9.000,7.52,7.500000,9.170,8.116200,2023,NaN,NaN
2227,RA-1001,Aluno-1001,0,10,-1,5.0,7.750000,9.347826,7.502,7.51,7.916667,7.920,7.796432,2024,-0.050000,-0.319768
1075,RA-1002,Aluno-1002,0,9,-1,5.0,7.000000,9.700000,9.000,7.52,6.250000,8.920,7.901200,2023,NaN,NaN


In [349]:
df["risco"] = (df["delta_INDE"] < -1).astype(int)

In [350]:
# ordenar por aluno e ano
df_shift = df.sort_values(["RA", "ANO"]).copy()

# criar alvo futuro (risco do ano seguinte)
df_shift["risco_futuro"] = (
    df_shift.groupby("RA")["risco"].shift(-1)
)

# remover linhas sem futuro
df_model = df_shift.dropna(subset=["risco_futuro"]).copy()

# garantir tipo inteiro
df_model["risco_futuro"] = df_model["risco_futuro"].astype(int)

print("Dataset temporal criado:", df_model.shape)
df_model.head()

Dataset temporal criado: (1369, 18)


,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE,risco,risco_futuro
0,RA-1,Aluno-1,0,19,-1,5.0,4.000000,4.1,8.300,5.60,7.50,7.278,5.783000,2022,NaN,NaN,0,0
1855,RA-1,Aluno-1,0,12,0,10.0,6.666667,8.6,8.751,7.50,7.50,7.583,7.388267,2023,2.666667,1.605267,0,0
1072,RA-1000,Aluno-1000,0,8,0,10.0,7.000000,9.4,8.500,3.77,6.25,8.920,7.916200,2023,NaN,NaN,0,0
1074,RA-1001,Aluno-1001,0,9,-1,5.0,7.800000,9.1,9.000,7.52,7.50,9.170,8.116200,2023,NaN,NaN,0,0
1075,RA-1002,Aluno-1002,0,9,-1,5.0,7.000000,9.7,9.000,7.52,6.25,8.920,7.901200,2023,NaN,NaN,0,0


In [351]:
features = [
    "SEXO", "IDADE", "DEFASAGEM",
    "IAN", "IDA", "IEG", "IAA",
    "IPS", "IPP", "IPV"
]

train_ano = 2022
test_ano  = 2023

X_train = df_model[df_model["ANO"] == train_ano][features]
y_train = df_model[df_model["ANO"] == train_ano]["risco_futuro"]

X_test = df_model[df_model["ANO"] == test_ano][features]
y_test = df_model[df_model["ANO"] == test_ano]["risco_futuro"]


In [352]:
X_train = X_train.fillna(X_train.median())
X_test  = X_test.fillna(X_train.median())  # usa estatística do treino


In [353]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

print("Acurácia:", model.score(X_test, y_test))


Acurácia: 0.8784313725490196


In [372]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.89      0.99      0.94       676
           1       0.25      0.02      0.04        89

    accuracy                           0.88       765
   macro avg       0.57      0.51      0.49       765
weighted avg       0.81      0.88      0.83       765



In [375]:
model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.25).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.95      0.92       676
           1       0.28      0.15      0.19        89

    accuracy                           0.86       765
   macro avg       0.59      0.55      0.56       765
weighted avg       0.82      0.86      0.84       765



In [376]:
for t in [0.5, 0.3, 0.2, 0.15, 0.1]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nLimiar: {t}")
    print(classification_report(y_test, y_pred))



Limiar: 0.5
              precision    recall  f1-score   support

           0       0.88      1.00      0.94       676
           1       0.00      0.00      0.00        89

    accuracy                           0.88       765
   macro avg       0.44      0.50      0.47       765
weighted avg       0.78      0.88      0.83       765


Limiar: 0.3
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       676
           1       0.22      0.06      0.09        89

    accuracy                           0.87       765
   macro avg       0.55      0.51      0.51       765
weighted avg       0.81      0.87      0.83       765


Limiar: 0.2
              precision    recall  f1-score   support

           0       0.89      0.90      0.90       676
           1       0.20      0.19      0.19        89

    accuracy                           0.82       765
   macro avg       0.55      0.54      0.55       765
weighted avg       0.81      0.82  

In [354]:
df_model.loc[df_model["ANO"] == test_ano, "prob_risco"] = (
    model.predict_proba(X_test)[:, 1]
)

In [355]:
importancias = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importancias)


IPV          0.203843
IDA          0.181354
IEG          0.159244
IDADE        0.119755
IAA          0.108354
IPS          0.095615
DEFASAGEM    0.067462
SEXO         0.034932
IAN          0.029441
IPP          0.000000
dtype: float64


In [356]:
df.head()

,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE,risco
0,RA-1,Aluno-1,0,19,-1,5.0,4.000000,4.1,8.300,5.6,7.5,7.278,5.783000,2022,NaN,NaN,0
1855,RA-1,Aluno-1,0,12,0,10.0,6.666667,8.6,8.751,7.5,7.5,7.583,7.388267,2023,2.666667,1.605267,0
2958,RA-1,Aluno-1,0,21,0,10.0,6.666667,0.0,8.751,7.5,7.5,7.583,7.388267,2024,0.000000,0.000000,0
9,RA-10,Aluno-10,0,18,-1,5.0,4.100000,5.2,8.300,5.0,7.5,7.056,5.784000,2022,NaN,NaN,0
99,RA-100,Aluno-100,0,13,1,10.0,7.600000,7.8,8.800,5.0,7.5,7.250,7.618000,2022,NaN,NaN,0


In [357]:
df_model.columns

Index(['RA', 'NOME', 'SEXO', 'IDADE', 'DEFASAGEM', 'IAN', 'IDA', 'IEG', 'IAA',
       'IPS', 'IPP', 'IPV', 'INDE', 'ANO', 'delta_IDA', 'delta_INDE', 'risco',
       'risco_futuro', 'prob_risco'],
      dtype='object')

In [358]:
df_model["prob_risco"].describe()

count    765.000000
mean       0.134148
std        0.115547
min        0.000000
25%        0.046667
50%        0.100000
75%        0.190000
max        0.560000
Name: prob_risco, dtype: float64

In [359]:
pd.cut(df_model["prob_risco"], bins=[0, .2, .4, .6, .8, 1]).value_counts()

prob_risco
(0.0, 0.2]    593
(0.2, 0.4]    139
(0.4, 0.6]     32
(0.6, 0.8]      0
(0.8, 1.0]      0
Name: count, dtype: int64

In [360]:
df_model[df_model["prob_risco"] > 0.8]

,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE,risco,risco_futuro,prob_risco


In [361]:
df_model["faixa_risco"] = pd.cut(
    df_model["prob_risco"].clip(0, 1),
    bins=[-0.001, .3, .7, 1],
    labels=["Baixo", "Médio", "Alto"]
)

In [362]:
df_model["faixa_risco"].value_counts(dropna=False)

faixa_risco
Baixo    687
NaN      604
Médio     78
Alto       0
Name: count, dtype: int64

In [363]:
df_model.sort_values("prob_risco", ascending=False).head(20)


,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE,risco,risco_futuro,prob_risco,faixa_risco
948,RA-922,Aluno-922,0,12,0,10.0,8.4,9.7,8.5,3.14,7.500000,10.010000,8.526200,2023,NaN,NaN,0,0,0.560000,Médio
985,RA-943,Aluno-943,0,12,0,10.0,8.0,9.4,9.5,7.52,7.500000,9.925000,8.917200,2023,NaN,NaN,0,0,0.553333,Médio
1391,RA-469,Aluno-469,0,12,-1,5.0,7.8,9.3,7.5,2.52,7.916667,8.843333,7.482533,2023,2.1,-0.141467,0,0,0.540000,Médio
1795,RA-38,Aluno-38,0,17,0,10.0,7.7,9.7,9.6,7.52,8.750000,8.882500,8.835333,2023,-0.9,0.312333,0,1,0.540000,Médio
1495,RA-183,Aluno-183,0,14,-1,5.0,7.7,9.7,7.9,2.52,7.968750,8.795000,7.586242,2023,3.9,0.286242,0,0,0.520000,Médio
1596,RA-176,Aluno-176,0,14,0,10.0,8.0,9.5,9.6,2.52,8.125000,9.002500,8.316833,2023,1.1,-0.110167,0,0,0.510000,Médio
867,RA-868,Aluno-868,1,12,0,10.0,7.1,9.5,10.0,8.76,7.187500,8.670000,8.638950,2023,NaN,NaN,0,1,0.506667,Médio
1301,RA-500,Aluno-500,0,11,0,10.0,7.9,10.0,9.5,2.52,8.958333,8.797500,8.427533,2023,-1.6,0.195533,0,0,0.503333,Médio
980,RA-789,Aluno-789,0,8,0,10.0,7.9,9.5,0.0,7.52,7.500000,10.010000,7.974000,2023,3.7,0.569000,0,0,0.500000,Médio
899,RA-693,Aluno-693,0,12,0,10.0,8.5,9.4,10.0,6.89,7.500000,10.010000,9.011200,2023,2.8,1.067200,0,0,0.486667,Médio


In [368]:
df_2024 = df[df["ANO"] == 2024]

X_2024 = df_2024[features].fillna(X_train.median())

df_2024 = df_2024.copy()
df_2024["prob_risco"] = model.predict_proba(X_2024)[:, 1]


In [369]:
df_2024["faixa_risco"] = pd.cut(
    df_2024["prob_risco"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["Muito baixo", "Baixo", "Médio", "Alto", "Crítico"]
)


In [370]:
df_2024.sort_values("prob_risco", ascending=False).head(20)

,RA,NOME,SEXO,IDADE,DEFASAGEM,IAN,IDA,IEG,IAA,IPS,IPP,IPV,INDE,ANO,delta_IDA,delta_INDE,risco,prob_risco,faixa_risco
2896,RA-56,Aluno-56,0,17,0,10.0,7.666667,9.265351,10.002,10.000,8.437500,9.233333,9.077020,2024,-0.133333,0.792687,0,0.833333,Crítico
2900,RA-87,Aluno-87,1,17,1,10.0,9.166667,9.649123,8.751,9.375,9.375000,9.623333,9.437925,2024,-0.133333,0.775425,0,0.620000,Alto
2796,RA-195,Aluno-195,0,15,1,10.0,7.833333,9.682540,7.500,6.260,7.968750,8.965000,8.469050,2024,1.833333,0.867000,0,0.606667,Alto
2449,RA-487,Aluno-487,0,12,0,10.0,9.666667,9.441270,9.168,9.380,9.062500,9.172500,9.417137,2024,0.166667,0.428271,0,0.606667,Alto
1959,RA-1321,Aluno-1321,1,7,0,10.0,8.750000,8.875000,9.002,6.885,6.562500,5.915000,7.952950,2024,NaN,NaN,0,0.576667,Médio
2798,RA-197,Aluno-197,0,14,0,10.0,8.333333,9.848485,7.500,6.260,7.968750,8.795000,8.568239,2024,1.133333,0.600630,0,0.556667,Médio
2689,RA-1081,Aluno-1081,0,15,0,10.0,8.833333,9.523810,6.667,5.005,9.166667,8.950000,8.545295,2024,2.333333,0.879262,0,0.536667,Médio
2465,RA-1096,Aluno-1096,0,11,1,10.0,8.500000,9.866667,9.168,8.755,9.062500,9.090000,9.189883,2024,-1.100000,0.574350,0,0.530000,Médio
2409,RA-816,Aluno-816,0,11,0,10.0,9.250000,9.772727,7.002,7.510,9.166667,8.786667,8.929745,2024,0.350000,0.372545,0,0.523333,Médio
2895,RA-45,Aluno-45,0,17,0,10.0,8.000000,9.619048,7.500,9.380,8.750000,8.283333,8.743476,2024,0.100000,0.439885,0,0.496667,Médio


In [371]:
df_2024["faixa_risco"].value_counts(dropna=False)

faixa_risco
Muito baixo    974
Baixo          136
Médio           25
NaN             17
Alto             3
Crítico          1
Name: count, dtype: int64

In [366]:
df_model.to_csv(r"C:\Users\gabri\Desktop\FIAP\Fase_5\Tech_Challenge\data\base_final.csv")

In [367]:
df_model.to_parquet(r"C:\Users\gabri\Desktop\FIAP\Fase_5\Tech_Challenge\data\base.parquet")